# Datamine COMPDH Process Example

This notebook demonstrates how to configure and run the **`compdh`** process wrapper in `dmstudio`.

## Process Description

## COMPDH

Composites drillhole data down or up each drillhole. By use of retrieval criteria and a very large compositing interval, **COMPDH** can also composite over rocktypes or seams.

Compositing can be performed either down the hole from the collar, or up the hole from the **EOH** position.

The input file must be in standard sample format (as output by process **DESURV**). The output file is in an identical format. Up to a maximum of 20 explicit numeric data fields may be composited. These do not have to be specified; they are identified by the process as those fields which are not the standard ones (**BHID** , **X** , **Y** , **Z** , **LENGTH** , **A0** , **B0** , **C0** , **RADIUS** , **FROM** , **TO**).

Each drillhole is split exactly into fixed length composites for a length equal to the parameter @**INTERVAL** , starting normally from the collar; if the optional parameter @**START** is set, this is the distance down the drillhole at which compositing will begin.

If there is a gap between samples of less than or equal to a specified distance (parameter @**MINGAP**) it will be ignored; that is, the missing part will be assigned the grades of the whole composite. Any gap greater than this, but less than or equal to the parameter @**MAXGAP** , will be replaced by a dummy sample with the default values specified in the file. A gap larger than @**MAXGAP** will be taken to terminate the composite.

If the total length of samples with non-absent grade values within a composite is greater than @**MINCOMP** , then the average grade of those samples is assigned to that grade field for the entire composite. If the total length of samples with non-absent grade values within a composite is less than @**MINCOMP** , then that grade field is assigned an absent data value for the entire composite. For example:

## FROM  |  TO |  AU

---|---|---
31.00 |  31.39 |  15.2
31.39 |  32.00 |  absent

If @**MINCOMP** =0.1, this is less than the assayed length of 0.39 and so the grade of 15.2 is assigned to the whole composite. If @**MINCOMP** =0.5, this is greater than the assayed length of 0.39 and so the absent data grade of - is assigned to the whole composite.

The file must be in the order of **BHID** and **FROM** , i.e. sorted in drillhole order in increasing downhole distance. This is the order output from the [DESURV](<desurv.md>) process.

### Weighting by Density

If a density field exists in the file then this may be used to for density weighted compositing. The density field is defined as the optional field * **DENSITY**. If any density value is absent data then the default density value will be used.

### Weighting by Core Loss or Recovery

To take into account the effects of core loss, the user may specify one of two optional fields * **CORELOSS** (core loss as a percentage) or * **COREREC** (core recovery as a percentage) to be used during compositing. The lost portion of the core will be taken into account and used in compositing. The actual treatment depends on the optional @**LOSS** parameter.

If @**LOSS** <=0 (default) then the lost part of the core will be assumed to have exactly the same grades, properties etc. as the recovered part. Core loss is ignored.

If @**LOSS** =1 then the lost core will be assumed to have default grades, density, properties etc. which will be averaged with the recovered core values. _Note: This requires that the drillhole file has default values for density & any other fields that need to be calculated._

If @**LOSS** >=2 then the lost core will be treated as cavity (zero density and grades) so that the grade of the total sample is effectively reduced by the cavity.

### Adjusting the Composite Interval

The parameter @**MODE** can be used to force equal composite lengths. If @**MODE** =0 (default) then part or all of one or more samples may be excluded from a composite. Setting @**MODE** =1 forces all samples to be included in one of the composites by adjusting the composite length.

For example if the sample data file contains 10 1m composites, and @**INTERVAL** =3 and the default @**MINCOMP** value of 1.5m is selected, then the output file will contain 3 3m composites. The final 1m sample will not be included in any composite. However if @**MODE** =1, then 3 composites each of length 3.333m will be created. If there were only 8 1m samples in the sample data file, then 3 composites of length 2.667m would be created.

### Residual Composites in COMPDH

**COMPDH** can be used to isolate and export samples excluded from composites (residual composites) using the &**RESIDUAL** output option, in combination with the compositing mode @**MODE** =0 or @**MODE** =2.

* @**MODE** =0 Samples are excluded if the final composite length is less than @MINCOMP. These excluded samples will appear in &**RESIDUAL**.

* @**MODE** =1 No residual file is created because all samples are included, ignoring any minimum length requirement.

* @**MODE** =2 Excluded samples are those whose length falls outside the allowable range set by [@**INTERVAL** \- @**RESLEN**] and [@**INTERVAL** \+ @**RESLEN**]. These samples appear in &**RESIDUAL**.

## #### @MODE=2

@MODE=2 allows the final composite in a given zone (defined by changes in BHID, *ZONE, *ZONE2, or *ZONE3 in a sorted drillhole file) to accommodate residual composites while still aiming for a target composite length @INTERVAL.

* All regular composites are formed at exactly @INTERVAL length, except possibly the last composite in each zone.

* The last composite can vary in length within the range [(@INTERVAL - @RESLEN), (@INTERVAL + @RESLEN)].

* For holes where reverse compositing is used (indicated by @REVERSE=1), the first composite of each zone may vary in length.

* Any composite that falls outside this range is excluded from the main output and listed in the &RESIDUAL file.

#### Using @RESLEN

The default value for @**RESLEN** is 0.5 * @**INTERVAL**. The maximum value is also 0.5 *@**INTERVAL**. If @**RESLEN** is left blank, the default option is used, which includes all residuals in &**OUT**.

If @**RESLEN** is less than 0.5 * @**INTERVAL** , any end-of-zone composite shorter than [@**INTERVAL** \- @**RESLEN**] is excluded from the &**OUT** file.

* **Note** (Residual composites allow a small over- or under-length for the final interval, ensuring minimal sample exclusion.):

### Dominant Values

COMPDH allows you to calculate up to 5 dominant values per composited interval. Fields DOM1 to DOM5 must exist in the input drillhole.

This can be useful where the input drillhole has flag fields and alpha fields which cannot be composited, so a length-weighted dominant value can be valuable information.

By default, a dominant field value must reach a minimum proportion of the total length of a composite that must share the same dominant value before that value is recorded. This is controlled by the **DOMPROPN** parameter, set to zero by default (always report the dominant value, regardless of proportion). Higher values require that the dominant values proportion exceeds the given threshold. If the dominant value within a composite is absent, the output value is absent regardless of the value of **DOMPROPN**.

### Input Files:

* **in** (Drillhole):
  Sample data file, sorted on **BHID** and **FROM**. Expects fields **BHID** , **FROM** ,

## **TO** , **LENGTH** , **X** , **Y** , **Z** , **A0** , **B0**.

  Required=Yes

### Output Files:

* **out** (Drillhole):
  Composite file.
  Required=Yes

* **residual** (Drillhole):
  Residual composites file.
  Required=No

### Fields:

* **bhid** (Any : IN):
  Drillhole identifier.
  Default=BHID
  Required=No

* **from** (Numeric : IN):
  Downhole distance to sample top.
  Default=FROM
  Required=No

* **to** (Numeric : IN):
  Downhole distance to sample bottom.
  Default=TO
  Required=No

* **density** (Numeric : IN):
  If present, composites will be density- weighted.
  Default=DENSITY
  Required=No

* **coreloss** (Numeric : IN):
  If present, will be taken as percentage core loss, and treated according to the
  **LOSS** parameter.
  Default=CORELOSS
  Required=No

* **corerec** (Numeric : IN):
  If present, will be taken as percentage core recovery, (100-core loss) and treated
  according to the **LOSS** parameter.
  Default=COREREC
  Required=No

* **zonen** (Any : IN):
  Name of a field for compositing within. (may be numeric or up to 32 character alpha).
  This field must exist in the IN and will be copied to the **OUT** file. If specified
  then new composites will be created each time the value of **ZONE** changes.Up to 5
  zones can be specified (**ZONE** to **ZONE5**).
  Default=Undefined
  Required=No

* **doms** (Undefined : Undefined):
  Name of a dominant field. If specified, this field is copied to the output file, containing the value that appears most frequently (by total length) within each sample. The field must exist in the IN file and can be numeric or up to 32 character alpha.Up to 5 dominant fields can be specified.
  Default=Undefined
  Required=No

### Parameters:

* **interval**:
  Composite length.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **mingap**:
  Gap length to be ignored. The default gap is calculated as 0.05 **INTERVAL**. This
  default value is applied if the parameter is not specified, or if the value is specified
  as <=0. A gap of exactly zero is not permitted. If you want the composite to be split at
  every gap, use a very small value for **MAXGAP** eg 0.0001.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxgap**:
  Gap length for termination of composite (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mincomp**:
  Minimum composite length [0.5 **INTERVAL**].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **loss**:
  If core loss or core recovery field is present, controls how it is handled: <=0 treat
  loss as part of sample =1 treat loss as default values >=2 treat as cavity [zero density
  and grades]
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **dompropn**:
  The minimum proportion of the total length of a composite that must share the same
  dominant value before that value is recorded. A value between 0 and 1. See Dominant
  Values.
  Range=
  Values=
  Default=
  Required=No

* **start**:
  Starting distance down hole (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mode**:
  If **MODE** is 0, the default, then the maximum composite length will be defined by the
  **INTERVAL** parameter and the minimum composite length by the **MINCOMP** parameter.
  Thus it is possible for part or all of one or more samples to be excluded from the
  composite. Setting **MODE** to 1 forces all samples to be included in one of the
  composites by adjusting the composite length, while keeping it as close as possible to
  **INTERVAL**. The maximum possible composite length will then be 1.5* **INTERVAL**.
  (0)If **MODE** is 2, excluded samples are those whose length falls outside the allowable
  range set by [@**INTERVAL** \- @**RESLEN**] and [@**INTERVAL** \+ @**RESLEN**]. These
  samples appear in &**RESIDUAL**.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **density**:
  Default density value of samples. This is used if no **DENSITY** field exists or if the
  **DENSITY** field contains absent values. If this is unset then the default **DENSITY**
  in the input file data definition is used as the default.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **reverse**:
  Reverse compositing direction. Control whether compositing is done from collar to toe
  (default) or toe to collar. 0 : Composite samples starting at the collar. This is the
  default. 1 : Composite samples from the toe to the collar.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:
  =3 to display each composite and output file DD (0).
  Range=0,3
  Values=0,1,2,3
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('compdh')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute compdh
print("Running compdh...")
dm_cmd.compdh(
    in_i='t_assays',  # required
    out_o='t_compdh_out',  # required
    interval_p=10,  # required
    # residual_o='t_compdh_out',  # optional
    # bhid_f='BHID',  # optional
    # from_f='FROM',  # optional
    # to_f='TO',  # optional
    # density_f='optional',  # optional
    # coreloss_f='optional',  # optional
    # corerec_f='optional',  # optional
    # zone_f='optional',  # optional
    # zone2_f='optional',  # optional
    # zone3_f='optional',  # optional
    # zone4_f='optional',  # optional
    # zone5_f='optional',  # optional
    # doms_f=['optional'],  # optional
    # mingap_p='optional',  # optional
    # maxgap_p=0,  # optional
    # mincomp_p='optional',  # optional
    # loss_p='optional',  # optional
    # dompropn_p='Required=No',  # optional
    # start_p=0,  # optional
    # mode_p=0,  # optional
    # density_p='optional',  # optional
    # reverse_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("compdh execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_compdh_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")